# NESTFUL Benchmark Evaluation — Google Colab

Evaluates **Qwen3-4B-Instruct** (baseline or fine-tuned LoRA) on the full  
**NESTFUL benchmark** (1 861 tasks) using the **official NESTFUL metrics** from the paper.

### Official metrics (from Table 1 of the NESTFUL paper)
| Metric | Description |
|--------|-------------|
| **F1 Func** | F1 over multiset of function names |
| **F1 Param** | F1 over multiset of (function, parameter) pairs |
| **Part. Acc.** | Partial sequence accuracy — fraction of aligned steps matching gold |
| **Full Acc.** | Full sequence accuracy — entire sequence matches gold exactly |
| **Win Rate** | Executable call sequence that leads to gold answer |

> **Note on Win Rate / Solution Equivalent:** The IBM executable functions are  
> not available on Colab. The executor runs in `gold_replay` mode.  
> F1, Part. Acc., and Full Acc. are fully reliable.  
> Win Rate reflects exact gold-trace execution only (not alternative paths).

### Data
Copy `nestful_data.jsonl` manually into the Colab session (see Cell 3),  
or let Cell 3 download it from `ibm-research/nestful` on HuggingFace.

### Runtime estimates
- T4 16 GB (free): ~2–3 h for 1 861 tasks, greedy, 4-bit
- A100 40 GB (Pro): ~40–60 min

In [ ]:
# ── Cell 1: GPU check ─────────────────────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
import torch
print(f"\nPyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  ({p.total_memory // 1024**2} MB)")

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
%pip install -q \
    transformers>=4.44 \
    accelerate>=0.33 \
    peft>=0.12 \
    bitsandbytes>=0.43 \
    pyyaml>=6.0 \
    tqdm>=4.66 \
    safetensors>=0.4 \
    datasets>=2.20
# vLLM — volitelně, ~5-10× rychlejší inference (Cell 5: USE_VLLM=True)
# %pip install -q vllm

print("Done.")

In [ ]:
# ── Cell 3: Mount Google Drive a nastavení cest ───────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

import os, sys

# Cesta ke zkopírované složce nestful_mtgrpo_minimal na Google Drive
WORK_DIR = "/content/drive/MyDrive/MT-GRPO/nestful_mtgrpo_minimal"

# Přidej WORK_DIR do Python path aby fungovaly importy (data.py, rollout.py, ...)
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)

os.chdir(WORK_DIR)
print(f"cwd: {os.getcwd()}")
print(f"Soubory: {sorted(f for f in os.listdir(WORK_DIR) if not f.startswith('.'))}")

In [ ]:
# ── Cell 4: NESTFUL data ──────────────────────────────────────────────────────
# Option A (recommended): copy nestful_data.jsonl manually.
#   Upload via Colab Files panel → /content/nestful_data.jsonl
#   OR mount Google Drive and point MANUAL_JSONL to it.
#
# Option B: download automatically from ibm-research/nestful on HuggingFace.
#   Set MANUAL_JSONL = None  to trigger download.

import json, pathlib

MANUAL_JSONL = None   # nastav cestu pokud jsi zkopíroval soubor jinam,
                      # jinak notebook hledá data přímo ve WORK_DIR

NESTFUL_JSONL = os.path.join(WORK_DIR, "data", "NESTFUL-main", "data_v2", "nestful_data.jsonl")

def _count_tasks(path):
    return sum(1 for l in open(path, encoding="utf-8") if l.strip())

# --- Check if data already exists in the repo clone ---
if os.path.isfile(NESTFUL_JSONL):
    n = _count_tasks(NESTFUL_JSONL)
    print(f"[data] Found in repo: {NESTFUL_JSONL}  ({n} tasks)")

# --- Option A: manually provided file ---
elif MANUAL_JSONL and os.path.isfile(MANUAL_JSONL):
    # Check if it already uses our format (has 'task_id') or HF format (has 'sample_id')
    sample = json.loads(open(MANUAL_JSONL, encoding="utf-8").readline())
    if "sample_id" in sample or "input" in sample:
        print("[data] HuggingFace format detected — converting to internal format...")
        rows = []
        for line in open(MANUAL_JSONL, encoding="utf-8"):
            if not line.strip(): continue
            ex = json.loads(line)
            # HF field mapping: sample_id→task_id, input→question, output→gold_calls
            gold_calls = ex.get("output") or ex.get("gold_calls") or []
            if isinstance(gold_calls, str):
                gold_calls = json.loads(gold_calls)
            tools = ex.get("tools") or []
            if isinstance(tools, str):
                tools = json.loads(tools)
            gold_answer = ex.get("gold_answer")
            if isinstance(gold_answer, str):
                try:
                    gold_answer = json.loads(gold_answer)
                except Exception:
                    try:
                        gold_answer = eval(gold_answer)   # HF docs say to use eval()
                    except Exception:
                        pass
            rows.append({
                "task_id":    ex.get("sample_id") or ex.get("task_id") or "",
                "question":   ex.get("input") or ex.get("question") or "",
                "gold_calls": gold_calls,
                "gold_answer": gold_answer,
                "num_calls":  len(gold_calls),
                "tools":      tools,
            })
        pathlib.Path(NESTFUL_JSONL).parent.mkdir(parents=True, exist_ok=True)
        with open(NESTFUL_JSONL, "w", encoding="utf-8") as fh:
            for r in rows:
                fh.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[data] Converted {len(rows)} tasks → {NESTFUL_JSONL}")
    else:
        # Already internal format, just symlink/copy
        import shutil
        pathlib.Path(NESTFUL_JSONL).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(MANUAL_JSONL, NESTFUL_JSONL)
        print(f"[data] Copied {_count_tasks(NESTFUL_JSONL)} tasks → {NESTFUL_JSONL}")

# --- Option B: auto-download from HuggingFace ---
else:
    print("[data] Downloading ibm-research/nestful from HuggingFace...")
    from datasets import load_dataset
    ds = load_dataset("ibm-research/nestful", split="train")  # only split is 'train'
    rows = []
    for ex in ds:
        gold_calls = ex.get("output") or []
        if isinstance(gold_calls, str):
            gold_calls = json.loads(gold_calls)
        tools = ex.get("tools") or []
        if isinstance(tools, str):
            tools = json.loads(tools)
        gold_answer = ex.get("gold_answer")
        if isinstance(gold_answer, str):
            try:
                gold_answer = eval(gold_answer)
            except Exception:
                pass
        rows.append({
            "task_id":    ex["sample_id"],
            "question":   ex["input"],
            "gold_calls": gold_calls,
            "gold_answer": gold_answer,
            "num_calls":  len(gold_calls),
            "tools":      tools,
        })
    pathlib.Path(NESTFUL_JSONL).parent.mkdir(parents=True, exist_ok=True)
    with open(NESTFUL_JSONL, "w", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"[data] Saved {len(rows)} tasks → {NESTFUL_JSONL}")

# Sanity check
from collections import Counter
rows = [json.loads(l) for l in open(NESTFUL_JSONL, encoding="utf-8") if l.strip()]
nc = Counter(r.get("num_calls", 0) for r in rows)
print(f"Total: {len(rows)} tasks")
print("Per num_calls:", dict(sorted(nc.items())))

In [ ]:
# ── Cell 5: Configuration ─────────────────────────────────────────────────────
# ADAPTER_PATH:
#   None          → baseline (base model, no fine-tuning)
#   "/path/..."   → evaluate a LoRA adapter (upload to Drive / Colab Files)

ADAPTER_PATH = None   # e.g. "/content/drive/MyDrive/MT-GRPO/nestful_mtgrpo_minimal/outputs/curriculum/stage_3/checkpoints/adapter_epoch_3"

# Výsledky jdou přímo na Drive — nevymažou se po skončení Colab session.
import datetime
_label = "baseline" if ADAPTER_PATH is None else "finetuned"
_ts    = datetime.datetime.now().strftime("%Y%m%d_%H%M")
OUTPUT_DIR = f"/content/drive/MyDrive/MT-GRPO/nestful_eval_{_label}_{_ts}"
MAX_TASKS    = None   # None = all 1861 tasks; set e.g. 30 for a quick smoke test

# NUM_ROLLOUTS: počet rolloutů na task.
#   1  → deterministický greedy (pro srovnání s NESTFUL paperm, temperature=0)
#   4  → stochastické rollouty (konzistentní s tréninkem a starým baseline)
#        binární metriky (Full Acc, Win Rate, strict) → pass@4 (MAX)
#        kontinuální metriky (F1 Func, F1 Param, Part. Acc.) → průměr
NUM_ROLLOUTS = 1

# Temperature: 0.0 = greedy (standard pro NESTFUL Table 1 baseline)
TEMPERATURE    = 0.0
MAX_NEW_TOKENS = 1024   # většina tasků stačí; 2048 je zbytečně pomalé

# vLLM na Colabu typicky NEFUNGUJE (PyPI wheel = CUDA 13, Colab = CUDA 12).
# Používej vLLM jen na DGX. Na Colabu nech False.
USE_VLLM = False

os.makedirs(OUTPUT_DIR, exist_ok=True)
label = "baseline" if ADAPTER_PATH is None else "finetuned"
print(f"Evaluating:   {label}")
print(f"Adapter:      {ADAPTER_PATH or 'None (base model)'}")
print(f"Tasks:        {MAX_TASKS or 'all (' + str(len(rows)) + ')'}")
print(f"Rollouts:     {NUM_ROLLOUTS}  (temperature={TEMPERATURE})")
print(f"Output:       {OUTPUT_DIR}")
if NUM_ROLLOUTS > 1:
    total = (MAX_TASKS or len(rows)) * NUM_ROLLOUTS
    print(f"Total episodes: {total}")

In [ ]:
# ── Cell 5b: Adapter z Google Drive (volitelné) ──────────────────────────────
# Drive je už mountnutý v Cell 3. Pokud chceš evaluovat fine-tuned adapter,
# odkomentuj a uprav ADAPTER_PATH. Cesta je přímo do Drive složky s checkpointem.

# ADAPTER_PATH = os.path.join(WORK_DIR, "outputs", "curriculum",
#                              "stage_3", "checkpoints", "adapter_epoch_3")
print(f"Adapter: {ADAPTER_PATH or 'None (base model baseline)'}")

In [ ]:
# ── Cell 6: Run evaluation ────────────────────────────────────────────────────
import subprocess, sys, shlex

# vLLM není v Colab defaultně — nainstaluj ho, nebo fallback na HF generate
_use_vllm = USE_VLLM
if _use_vllm:
    try:
        from vllm import LLM  # noqa: F401 — musí projít i CUDA extension
        print("vLLM OK.")
    except ImportError as e:
        print(f"vLLM unavailable ({e}) — using HF generate.")
        _use_vllm = False

overrides = [
    f"paths.full_nestful_jsonl={NESTFUL_JSONL}",
    f"paths.eval_jsonl={NESTFUL_JSONL}",
    f"experiment.output_dir={OUTPUT_DIR}",
    f"hardware.use_vllm={'true' if _use_vllm else 'false'}",
    "hardware.load_in_4bit=false",   # A100 40GB: bf16 je rychlejší
    "hardware.bf16=true",
    "finetuning.load_in_4bit=false", # nutné — jinak config.yaml přepíše na 4bit
    "hardware.gradient_checkpointing=false",
    f"generation.temperature={TEMPERATURE}",
    f"generation.max_new_tokens={MAX_NEW_TOKENS}",
    f"generation.max_new_tokens_eval={MAX_NEW_TOKENS}",
    f"generation.top_p={'1.0' if TEMPERATURE == 0 else '0.95'}",
    "logging.use_wandb=false",
    "data.eval_stage=null",   # all call counts, no filter
    f"data.num_eval_rollouts={NUM_ROLLOUTS}",
]
if MAX_TASKS is not None:
    overrides.append(f"data.max_eval_tasks={MAX_TASKS}")

cmd = [
    sys.executable,
    os.path.join(WORK_DIR, "run.py"),
    "--mode", "final_eval",
    "--config", os.path.join(WORK_DIR, "config.yaml"),
]
if ADAPTER_PATH:
    cmd += ["--checkpoint", ADAPTER_PATH]
for ov in overrides:
    cmd += ["--override", ov]

print("Command:")
print(" ".join(shlex.quote(c) for c in cmd))
print("\n" + "─"*70)

proc = subprocess.Popen(
    cmd, cwd=WORK_DIR,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nExited: {proc.returncode}")

In [ ]:
# ── Cell 7: Display results — official NESTFUL metrics ────────────────────────
import json, os

# final_eval writes to final_eval_metrics.json
metrics_path = os.path.join(OUTPUT_DIR, "final_eval_metrics.json")
if not os.path.isfile(metrics_path):
    metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")

if not os.path.isfile(metrics_path):
    print(f"No metrics file found in {OUTPUT_DIR} — did the eval complete?")
else:
    m = json.load(open(metrics_path, encoding="utf-8"))
    off = m.get("nestful_official_compatible", {})
    ours = m.get("our_metrics", {})
    limited = not m.get("solution_equivalent_reportable", False)

    print("╔══ NESTFUL Official Metrics (Table 1 from paper) ══════════════════╗")
    print(f"  Model:    Qwen3-4B-Instruct  {'(base)' if not ADAPTER_PATH else '(fine-tuned)'}")
    print(f"  Tasks:    {m.get('num_tasks', '?')}  |  Executor: {m.get('executor_mode', '?')}")
    if limited:
        print("  ⚠ gold_replay mode: Win Rate reflects gold-trace only (not alternative paths)")
    print("├──────────────────────────────────────────────────────────────────┤")
    print(f"  F1 Func   (f1_func)           : {off.get('f1_func', 0):.4f}")
    print(f"  F1 Param  (f1_param)          : {off.get('f1_param', 0):.4f}")
    print(f"  Part. Acc (partial_seq_acc)   : {off.get('partial_sequence_accuracy', 0):.4f}")
    print(f"  Full Acc  (full_seq_acc)      : {off.get('full_sequence_accuracy', 0):.4f}")
    print(f"  Win Rate                      : {off.get('win_rate', 0):.4f}  {'[limited]' if limited else ''}")
    print("├── Our additional metrics ─────────────────────────────────────────┤")
    print(f"  strict_gold_trace_pass        : {ours.get('strict_gold_trace_pass', 0):.4f}")
    print(f"  final_answer_pass             : {ours.get('final_answer_pass', 0):.4f}")
    print(f"  solution_equivalent_pass      : {ours.get('solution_equivalent_pass', 0):.4f}  {'[limited]' if limited else ''}")
    print(f"  zero_tool_calls               : {ours.get('zero_tool_calls', m.get('zero_tool_calls', 0)):.4f}")
    print("╚══════════════════════════════════════════════════════════════════╝")

    # --- Comparison with published paper numbers (One-shot ICL, Table 1) ---
    paper_baselines = {
        # Model name: (F1 Func, F1 Param, Part. Acc, Full Acc, Win Rate)
        "llama-3-1-8b-instruct": (0.58, 0.37, 0.24, 0.18, 0.16),
        "granite-3.0-8b-instruct": (0.50, 0.27, 0.15, 0.05, 0.07),
        "xLAM-7b-fc-r": (0.54, 0.32, 0.18, 0.10, 0.13),
        "MadeAgents/Hammer2.0-7b (best 7B)": (0.62, 0.42, 0.29, 0.25, 0.33),
    }
    our_row = (
        off.get('f1_func', 0),
        off.get('f1_param', 0),
        off.get('partial_sequence_accuracy', 0),
        off.get('full_sequence_accuracy', 0),
        off.get('win_rate', 0),
    )
    print("\n── Comparison: One-shot ICL from Table 1 ──────────────────────────")
    print(f"{'Model':<40} {'F1F':>5} {'F1P':>5} {'PAcc':>5} {'FAcc':>5} {'WR':>5}")
    print("-" * 65)
    for name, vals in paper_baselines.items():
        print(f"{name:<40} {vals[0]:>5.2f} {vals[1]:>5.2f} {vals[2]:>5.2f} {vals[3]:>5.2f} {vals[4]:>5.2f}")
    print("-" * 65)
    model_label = f"Qwen3-4B {'(base)' if not ADAPTER_PATH else '(FT)'}"
    print(f"{model_label:<40} {our_row[0]:>5.2f} {our_row[1]:>5.2f} {our_row[2]:>5.2f} {our_row[3]:>5.2f} {our_row[4]:>5.2f}")

In [ ]:
# ── Cell 8: Per-num-calls breakdown ──────────────────────────────────────────
import json, os

metrics_path = os.path.join(OUTPUT_DIR, "final_eval_metrics.json")
if not os.path.isfile(metrics_path):
    metrics_path = os.path.join(OUTPUT_DIR, "metrics.json")

if os.path.isfile(metrics_path):
    m = json.load(open(metrics_path, encoding="utf-8"))
    by_n = m.get("by_num_calls", {})
    if by_n:
        print(f"{'n':>3} | {'tasks':>5} | {'F1F':>5} | {'F1P':>5} | {'PAcc':>6} | {'FAcc':>6} | {'WR':>5} | {'strict':>6}")
        print("-" * 65)
        for nc_key in sorted(by_n, key=int):
            d = by_n[nc_key]
            off = d.get("nestful_official_compatible", {})
            ours = d.get("our_metrics", {})
            count = d.get("count", "?")
            print(f"{nc_key:>3} | {str(count):>5} | "
                  f"{off.get('f1_func',0):>5.3f} | "
                  f"{off.get('f1_param',0):>5.3f} | "
                  f"{off.get('partial_sequence_accuracy',0):>6.3f} | "
                  f"{off.get('full_sequence_accuracy',0):>6.3f} | "
                  f"{off.get('win_rate',0):>5.3f} | "
                  f"{ours.get('strict_gold_trace_pass',0):>6.3f}")
    else:
        print("No per-num-calls breakdown available — computing from trajectories...")

        from collections import defaultdict
        traj_path = os.path.join(OUTPUT_DIR, "final_eval_trajectories.jsonl")
        if not os.path.isfile(traj_path):
            traj_path = os.path.join(OUTPUT_DIR, "rollout_eval_trajectories.jsonl")
        if os.path.isfile(traj_path):
            rows_t = [json.loads(l) for l in open(traj_path, encoding="utf-8") if l.strip()]
            by_n2 = defaultdict(lambda: {"strict": [], "final": [], "zero": [], "f1f": [], "pacc": [], "facc": []})
            for r in rows_t:
                n = r.get("num_gold_calls", r.get("gold_n", 0))
                d = r.get("diagnostics", {})
                off2 = r.get("official_metrics", {})
                by_n2[n]["strict"].append(float(r.get("reward_train_strict", 0)))
                by_n2[n]["final"].append(1.0 if d.get("final_answer_pass") else 0.0)
                by_n2[n]["zero"].append(1.0 if d.get("zero_tool_calls") else 0.0)
            mean = lambda v: sum(v)/len(v) if v else 0.0
            print(f"{'n':>3} | {'tasks':>5} | {'strict':>6} | {'final_ans':>9} | {'zero_tool':>9}")
            print("-" * 45)
            totals = defaultdict(list)
            for n in sorted(by_n2):
                d = by_n2[n]
                print(f"{n:>3} | {len(d['strict']):>5} | {mean(d['strict']):>6.4f} | {mean(d['final']):>9.4f} | {mean(d['zero']):>9.4f}")
                for k in d: totals[k].extend(d[k])
            print("-" * 45)
            print(f"{'ALL':>3} | {len(totals['strict']):>5} | {mean(totals['strict']):>6.4f} | {mean(totals['final']):>9.4f} | {mean(totals['zero']):>9.4f}")

In [ ]:
# ── Cell 9: Výsledky jsou na Drive ───────────────────────────────────────────
# OUTPUT_DIR byl nastaven přímo na Google Drive v Cell 5,
# takže všechny soubory (metrics.json, trajectories.jsonl) jsou uloženy tam.
import os, json
print(f"Výsledky uloženy v: {OUTPUT_DIR}")
print(f"Soubory: {sorted(os.listdir(OUTPUT_DIR)) if os.path.isdir(OUTPUT_DIR) else 'složka neexistuje'}")
mpath = os.path.join(OUTPUT_DIR, 'metrics.json')
if os.path.isfile(mpath):
    m = json.load(open(mpath))
    off = m.get('nestful_official_compatible', {})
    print(f"\nQuick summary:")
    print(f"  F1 Func:   {off.get('f1_func', 0):.4f}")
    print(f"  Full Acc:  {off.get('full_sequence_accuracy', 0):.4f}")
    print(f"  Win Rate:  {off.get('win_rate', 0):.4f}")